# Multi-Label Rice Leaf Disease Classification — EfficientNet-B4 (CNN)

**Mirrors your teammate's SVM experiment design, so results are directly comparable.**

**Goal:** Train on single-disease leaf images (BLB-only, Blast-only, BrownSpot-only), then test
whether the model can correctly name **all diseases present** on composite (co-infected) leaf
images it never saw during training.

**Key differences from the SVM version:**
- SVM used a **frozen** ResNet50 feature extractor + separate per-label SVMs on top.
  This CNN version **fine-tunes EfficientNet-B4 end-to-end** with a single multi-label sigmoid head,
  so the features themselves adapt to distinguishing co-occurring lesion types.
- SVM needed a lesion-box-proposal step + classified each box separately, because a frozen-feature
  SVM pipeline has no way to naturally handle "multiple things in one image." A CNN with a
  multi-label output can be run directly on the **whole composite image** — no box proposals needed.

Edit the paths in Cell 1 to match your Drive structure, then Run All.

## Step 1: Setup + paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, time, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, hamming_loss, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# ---- EDIT THESE PATHS ----
BASE_DIR = '/content/drive/MyDrive'

# single-disease-only folders (used for TRAINING)
BLB_DIR        = os.path.join(BASE_DIR, 'enhanced_dataset/bacterial_leaf_blight')
BLAST_DIR      = os.path.join(BASE_DIR, 'enhanced_dataset/blast')
BROWN_SPOT_DIR = os.path.join(BASE_DIR, 'enhanced_dataset/brown_spot')

# composite / co-infected folders (used for TESTING generalization only, never for training)
COMPOSITE_DIRS = {
    'blb_on_brownspot':   os.path.join(BASE_DIR, 'augmentation/blb_on_brown_spot_augmented'),
    'blast_on_brownspot': os.path.join(BASE_DIR, 'augmentation/blast_on_brown_spot_augmented'),
    'blb_on_blast':        os.path.join(BASE_DIR, 'augmentation/blb_on_blast_augmented'),
}

MODEL_OUT = os.path.join(BASE_DIR, 'effnetb4_multilabel_model.h5')

CLASS_NAMES = ['BLB', 'Blast', 'BrownSpot']   # fixed label order used throughout this notebook
N_PER_CLASS = 900   # cap per single-disease class to keep training balanced (matches teammate's SVM setup)
SEED = 42
random.seed(SEED)

for name, path in [('BLB', BLB_DIR), ('Blast', BLAST_DIR), ('BrownSpot', BROWN_SPOT_DIR)]:
    exists = os.path.isdir(path)
    n = len(glob.glob(path + '/*')) if exists else 0
    print(f"{name:12s} single-disease -> exists={exists}, num_images={n}")

for name, path in COMPOSITE_DIRS.items():
    exists = os.path.isdir(path)
    n = len(glob.glob(path + '/*')) if exists else 0
    print(f"{name:22s} composite -> exists={exists}, num_images={n}")

Mounted at /content/drive
TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
BLB          single-disease -> exists=True, num_images=1079
Blast        single-disease -> exists=True, num_images=2760
BrownSpot    single-disease -> exists=True, num_images=1392
blb_on_brownspot       composite -> exists=True, num_images=900
blast_on_brownspot     composite -> exists=True, num_images=900
blb_on_blast           composite -> exists=True, num_images=900


**Check this output before continuing.** All 6 folders should show `exists=True` and `num_images > 0`. If BLB_DIR doesn't exist yet, either point it to your actual BLB-only folder, or remove BLB from `CLASS_NAMES` and adjust the label-building cells below to use only 2 labels (blast, brown_spot) like the SVM version.

## Step 2: Build training dataframe from the 3 single-disease folders

In [2]:
def sample_paths(folder, n):
    all_paths = sorted(glob.glob(folder + '/*'))
    return random.sample(all_paths, min(n, len(all_paths)))

blb_paths   = sample_paths(BLB_DIR, N_PER_CLASS)
blast_paths = sample_paths(BLAST_DIR, N_PER_CLASS)
brown_paths = sample_paths(BROWN_SPOT_DIR, N_PER_CLASS)

print(f"Sampled: BLB={len(blb_paths)}, Blast={len(blast_paths)}, BrownSpot={len(brown_paths)}")

rows = []
for p in blb_paths:
    rows.append({'filepath': p, 'BLB': 1, 'Blast': 0, 'BrownSpot': 0})
for p in blast_paths:
    rows.append({'filepath': p, 'BLB': 0, 'Blast': 1, 'BrownSpot': 0})
for p in brown_paths:
    rows.append({'filepath': p, 'BLB': 0, 'Blast': 0, 'BrownSpot': 1})

train_full_df = pd.DataFrame(rows)
print("\nTotal single-disease training images:", len(train_full_df))
print(train_full_df[CLASS_NAMES].sum())
train_full_df.head()

Sampled: BLB=900, Blast=900, BrownSpot=900

Total single-disease training images: 2700
BLB          900
Blast        900
BrownSpot    900
dtype: int64


,filepath,BLB,Blast,BrownSpot
0,/content/drive/MyDrive/enhanced_dataset/bacter...,1,0,0
1,/content/drive/MyDrive/enhanced_dataset/bacter...,1,0,0
2,/content/drive/MyDrive/enhanced_dataset/bacter...,1,0,0
3,/content/drive/MyDrive/enhanced_dataset/bacter...,1,0,0
4,/content/drive/MyDrive/enhanced_dataset/bacter...,1,0,0


## Step 3: Train/validation split (held-out single-disease images, for Step 9's sanity check)

In [3]:
strat_key = train_full_df[CLASS_NAMES].astype(str).agg('-'.join, axis=1)
train_df, val_df = train_test_split(
    train_full_df, test_size=0.15, random_state=SEED, stratify=strat_key
)
print("Train:", len(train_df), " Val (held-out single-disease):", len(val_df))

Train: 2295  Val (held-out single-disease): 405


## Step 4: Data generators (EfficientNet-B4 expects 380x380 input)

In [4]:
IMG_SIZE = 380
BATCH_SIZE = 16   # drop to 8 if you hit OOM on Colab's free T4

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col=CLASS_NAMES,
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='raw', shuffle=True   # 'raw' -> gives the binary label vector directly, needed for multi-label
)

val_gen = val_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col=CLASS_NAMES,
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='raw', shuffle=False
)

Found 2295 validated image filenames.
Found 405 validated image filenames.


## Step 5: Build the EfficientNet-B4 model (multi-label sigmoid head)

In [5]:
base_model = EfficientNetB4(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False  # freeze for stage 1

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)

# sigmoid, one neuron per disease -> each disease is an independent probability,
# this is what lets the model predict MULTIPLE diseases at once on a composite image
outputs = layers.Dense(len(CLASS_NAMES), activation='sigmoid')(x)

model = models.Model(inputs, outputs)
model.summary()

71686520/71686520 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 380, 380, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb4 (Functional)     │ (None, 12, 12, 1792)   │    17,673,823 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1792)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1792)           │         7,168 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       459,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,140,770 (69.20 MB)

 Trainable params: 463,363 (1.77 MB)

 Non-trainable params: 17,677,407 (67.43 MB)

## Step 6: Compile + callbacks

In [6]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['binary_accuracy', tf.keras.metrics.AUC(name='auc', multi_label=True)]
)

cb_list = [
    callbacks.ModelCheckpoint(MODEL_OUT, monitor='val_auc', mode='max', save_best_only=True),
    callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]

## Step 7: Train — Stage 1 (frozen base)

In [7]:
history_stage1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=cb_list
)

Epoch 1/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - auc: 0.5114 - binary_accuracy: 0.5754 - loss: 0.8645

144/144 ━━━━━━━━━━━━━━━━━━━━ 1537s 10s/step - auc: 0.5193 - binary_accuracy: 0.5893 - loss: 0.8503 - val_auc: 0.6858 - val_binary_accuracy: 0.6675 - val_loss: 0.6657 - learning_rate: 0.0010
Epoch 2/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 863ms/step - auc: 0.5387 - binary_accuracy: 0.6098 - loss: 0.7630

144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 957ms/step - auc: 0.5378 - binary_accuracy: 0.6118 - loss: 0.7560 - val_auc: 0.7282 - val_binary_accuracy: 0.6708 - val_loss: 0.6734 - learning_rate: 0.0010
Epoch 3/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 891ms/step - auc: 0.5459 - binary_accuracy: 0.6204 - loss: 0.7024

144/144 ━━━━━━━━━━━━━━━━━━━━ 141s 982ms/step - auc: 0.5369 - binary_accuracy: 0.6176 - loss: 0.7099 - val_auc: 0.7593 - val_binary_accuracy: 0.6716 - val_loss: 0.6791 - learning_rate: 0.0010
Epoch 4/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 956ms/step - auc: 0.5476 - binary_accuracy: 0.6338 - loss: 0.6796 - val_auc: 0.6176 - val_binary_accuracy: 0.6683 - val_loss: 0.6811 - learning_rate: 0.0010
Epoch 5/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 136s 938ms/step - auc: 0.5622 - binary_accuracy: 0.6436 - loss: 0.6547 - val_auc: 0.6805 - val_binary_accuracy: 0.6798 - val_loss: 0.6753 - learning_rate: 5.0000e-04
Epoch 6/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 134s 933ms/step - auc: 0.5626 - binary_accuracy: 0.6533 - loss: 0.6468 - val_auc: 0.7112 - val_binary_accuracy: 0.6749 - val_loss: 0.6705 - learning_rate: 5.0000e-04
Epoch 7/15
144/144 ━━━━━━━━━━━━━━━━━━━━ 136s 941ms/step - auc: 0.5731 - binary_accuracy: 0.6537 - loss: 0.6410 - val_auc: 0.7463 - val_binary_accuracy: 0.6774 - val_loss: 0.6689 - learning_rate: 

## Step 8: Train — Stage 2 (fine-tune top layers)

In [8]:
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['binary_accuracy', tf.keras.metrics.AUC(name='auc', multi_label=True)]
)

history_stage2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=cb_list
)

Epoch 1/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - auc: 0.5438 - binary_accuracy: 0.6056 - loss: 0.7822

144/144 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - auc: 0.5544 - binary_accuracy: 0.6150 - loss: 0.7635 - val_auc: 0.7663 - val_binary_accuracy: 0.6667 - val_loss: 0.7420 - learning_rate: 1.0000e-05
Epoch 2/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 935ms/step - auc: 0.5729 - binary_accuracy: 0.6286 - loss: 0.7341

144/144 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - auc: 0.5892 - binary_accuracy: 0.6325 - loss: 0.7181 - val_auc: 0.7750 - val_binary_accuracy: 0.6848 - val_loss: 0.8221 - learning_rate: 1.0000e-05
Epoch 3/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 962ms/step - auc: 0.6040 - binary_accuracy: 0.6504 - loss: 0.7113

144/144 ━━━━━━━━━━━━━━━━━━━━ 158s 1s/step - auc: 0.6209 - binary_accuracy: 0.6565 - loss: 0.6928 - val_auc: 0.7972 - val_binary_accuracy: 0.7053 - val_loss: 0.7498 - learning_rate: 1.0000e-05
Epoch 4/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 944ms/step - auc: 0.6179 - binary_accuracy: 0.6524 - loss: 0.7024

144/144 ━━━━━━━━━━━━━━━━━━━━ 151s 1s/step - auc: 0.6172 - binary_accuracy: 0.6465 - loss: 0.7033 - val_auc: 0.8066 - val_binary_accuracy: 0.7243 - val_loss: 0.5446 - learning_rate: 5.0000e-06
Epoch 5/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - auc: 0.6421 - binary_accuracy: 0.6606 - loss: 0.6679 - val_auc: 0.8062 - val_binary_accuracy: 0.7416 - val_loss: 0.5211 - learning_rate: 5.0000e-06
Epoch 6/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 913ms/step - auc: 0.6391 - binary_accuracy: 0.6600 - loss: 0.6763

...

144/144 ━━━━━━━━━━━━━━━━━━━━ 142s 989ms/step - auc: 0.6857 - binary_accuracy: 0.6793 - loss: 0.6241 - val_auc: 0.8273 - val_binary_accuracy: 0.7580 - val_loss: 0.5151 - learning_rate: 2.5000e-06
Epoch 19/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 896ms/step - auc: 0.6908 - binary_accuracy: 0.6960 - loss: 0.6215

144/144 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - auc: 0.7006 - binary_accuracy: 0.6950 - loss: 0.6110 - val_auc: 0.8274 - val_binary_accuracy: 0.7547 - val_loss: 0.5149 - learning_rate: 1.2500e-06
Epoch 20/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 904ms/step - auc: 0.6946 - binary_accuracy: 0.6871 - loss: 0.6193

144/144 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - auc: 0.6909 - binary_accuracy: 0.6835 - loss: 0.6214 - val_auc: 0.8282 - val_binary_accuracy: 0.7580 - val_loss: 0.5154 - learning_rate: 1.2500e-06
Epoch 21/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 897ms/step - auc: 0.7085 - binary_accuracy: 0.6936 - loss: 0.6081

144/144 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - auc: 0.7068 - binary_accuracy: 0.6962 - loss: 0.6073 - val_auc: 0.8289 - val_binary_accuracy: 0.7547 - val_loss: 0.5221 - learning_rate: 1.2500e-06
Epoch 22/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 892ms/step - auc: 0.6957 - binary_accuracy: 0.6878 - loss: 0.6166

144/144 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - auc: 0.7074 - binary_accuracy: 0.6924 - loss: 0.6038 - val_auc: 0.8297 - val_binary_accuracy: 0.7547 - val_loss: 0.5192 - learning_rate: 6.2500e-07
Epoch 23/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 959ms/step - auc: 0.6903 - binary_accuracy: 0.6816 - loss: 0.6254 - val_auc: 0.8294 - val_binary_accuracy: 0.7564 - val_loss: 0.5172 - learning_rate: 6.2500e-07
Epoch 24/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 132s 915ms/step - auc: 0.7002 - binary_accuracy: 0.6934 - loss: 0.6122 - val_auc: 0.8289 - val_binary_accuracy: 0.7523 - val_loss: 0.5132 - learning_rate: 6.2500e-07
Epoch 25/25
144/144 ━━━━━━━━━━━━━━━━━━━━ 141s 906ms/step - auc: 0.6961 - binary_accuracy: 0.6882 - loss: 0.6184 - val_auc: 0.8295 - val_binary_accuracy: 0.7547 - val_loss: 0.5184 - learning_rate: 3.1250e-07


## Step 9: Sanity-check report on held-out single-disease images\nThis is the CNN equivalent of the SVM notebook's Cell 5/6 per-label report — same data type (single-disease, seen distribution), just to confirm training worked before testing on composites.

In [9]:
val_gen.reset()
y_pred_probs_val = model.predict(val_gen)
y_pred_val = (y_pred_probs_val > 0.5).astype(int)
y_true_val = val_df[CLASS_NAMES].values

print("=== Per-label report on held-out single-disease images ===")
print(classification_report(y_true_val, y_pred_val, target_names=CLASS_NAMES, zero_division=0))

26/26 ━━━━━━━━━━━━━━━━━━━━ 38s 960ms/step
=== Per-label report on held-out single-disease images ===
              precision    recall  f1-score   support

         BLB       0.66      0.90      0.76       135
       Blast       0.87      0.30      0.44       135
   BrownSpot       0.63      0.25      0.36       135

   micro avg       0.69      0.48      0.57       405
   macro avg       0.72      0.48      0.52       405
weighted avg       0.72      0.48      0.52       405
 samples avg       0.48      0.48      0.48       405



## Step 10: THE REAL TEST — evaluate directly on composite (co-infected) images

This is the CNN equivalent of the SVM notebook's Cell 9 (`detect_labels_present` + final report).
Ground truth for each composite image is derived automatically from which folder it came from —
e.g. `blb_on_brownspot` images have ground truth `[1, 0, 1]` (BLB present, Blast absent, BrownSpot present).

**No box-proposal step needed** — we feed the whole composite image straight into the model,
since the multi-label sigmoid head can already flag multiple diseases from one pass.

In [10]:
# ground-truth vector per composite folder, based on which 2 diseases that folder represents
FOLDER_TRUE_LABELS = {
    'blb_on_brownspot':   [1, 0, 1],   # BLB + BrownSpot
    'blast_on_brownspot': [0, 1, 1],   # Blast + BrownSpot
    'blb_on_blast':        [1, 1, 0],   # BLB + Blast
}

valid_ext = ('.jpg', '.jpeg', '.png', '.bmp')
composite_rows = []
for folder_label, folder in COMPOSITE_DIRS.items():
    for fname in os.listdir(folder):
        if fname.lower().endswith(valid_ext):
            row = {'filepath': os.path.join(folder, fname), 'source_folder': folder_label}
            for i, cls in enumerate(CLASS_NAMES):
                row[cls] = FOLDER_TRUE_LABELS[folder_label][i]
            composite_rows.append(row)

composite_df = pd.DataFrame(composite_rows)
print("Total composite test images:", len(composite_df))
print(composite_df.groupby('source_folder').size())

Total composite test images: 2700
source_folder
blast_on_brownspot    900
blb_on_blast          900
blb_on_brownspot      900
dtype: int64


In [11]:
composite_datagen = ImageDataGenerator(rescale=1./255)
composite_gen = composite_datagen.flow_from_dataframe(
    composite_df, x_col='filepath', y_col=CLASS_NAMES,
    target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='raw', shuffle=False
)

y_pred_probs_comp = model.predict(composite_gen)
y_pred_comp = (y_pred_probs_comp > 0.5).astype(int)
y_true_comp = composite_df[CLASS_NAMES].values

print("=== Per-label report on COMPOSITE (co-infected, unseen) images ===")
print(classification_report(y_true_comp, y_pred_comp, target_names=CLASS_NAMES, zero_division=0))

exact_match = accuracy_score(y_true_comp, y_pred_comp)
h_loss = hamming_loss(y_true_comp, y_pred_comp)
print(f"Exact match ratio (all diseases correctly identified together): {exact_match:.3f}")
print(f"Hamming loss: {h_loss:.3f}")
for i, cls in enumerate(CLASS_NAMES):
    print(f"{cls} detected: {y_pred_comp[:, i].mean() * 100:.1f}%  (true rate: {y_true_comp[:, i].mean() * 100:.1f}%)")

Found 2700 validated image filenames.
169/169 ━━━━━━━━━━━━━━━━━━━━ 1606s 10s/step
=== Per-label report on COMPOSITE (co-infected, unseen) images ===
              precision    recall  f1-score   support

         BLB       0.72      0.20      0.32      1800
       Blast       0.78      0.22      0.35      1800
   BrownSpot       0.80      0.16      0.26      1800

   micro avg       0.76      0.19      0.31      5400
   macro avg       0.77      0.19      0.31      5400
weighted avg       0.77      0.19      0.31      5400
 samples avg       0.37      0.19      0.25      5400

Exact match ratio (all diseases correctly identified together): 0.004
Hamming loss: 0.578
BLB detected: 19.0%  (true rate: 66.7%)
Blast detected: 19.0%  (true rate: 66.7%)
BrownSpot detected: 13.0%  (true rate: 66.7%)


## Step 11: Per-combination breakdown

Since your 3 composite folders each stress a different disease *pair*, break the results down
by folder to directly compare against what you'd expect from the SVM's blast-vs-brownspot numbers.

In [12]:
for folder_label in COMPOSITE_DIRS.keys():
    mask = (composite_df['source_folder'] == folder_label).values
    if mask.sum() == 0:
        continue
    print(f"\n--- {folder_label} (n={mask.sum()}) ---")
    print(classification_report(
        y_true_comp[mask], y_pred_comp[mask],
        target_names=CLASS_NAMES, zero_division=0
    ))


--- blb_on_brownspot (n=900) ---
              precision    recall  f1-score   support

         BLB       1.00      0.15      0.27       900
       Blast       0.00      0.00      0.00         0
   BrownSpot       1.00      0.16      0.27       900

   micro avg       0.71      0.15      0.25      1800
   macro avg       0.67      0.10      0.18      1800
weighted avg       1.00      0.15      0.27      1800
 samples avg       0.30      0.15      0.20      1800


--- blast_on_brownspot (n=900) ---
              precision    recall  f1-score   support

         BLB       0.00      0.00      0.00         0
       Blast       1.00      0.13      0.23       900
   BrownSpot       1.00      0.16      0.27       900

   micro avg       0.64      0.14      0.24      1800
   macro avg       0.67      0.10      0.17      1800
weighted avg       1.00      0.14      0.25      1800
 samples avg       0.28      0.14      0.19      1800


--- blb_on_blast (n=900) ---
              precision    rec

## What to expect / how to read this

- **Compare Step 9 vs Step 10:** if Step 9 (held-out single-disease) looks strong but Step 10
  (composite) drops sharply, that tells you the model isn't generalizing to co-occurring lesions —
  the same failure mode the SVM had, just hopefully less severe since the CNN is fine-tuned
  end-to-end rather than relying on frozen features + independent per-label classifiers.
- **Watch for an imbalanced per-label result** like the SVM's (blast recall 0.33 vs brown_spot
  recall 0.95) — if one disease's lesions are visually more dominant or a different scale/color,
  the sigmoid head may still favor it. If that shows up, `class_weight`-style loss weighting per
  label (not shown here since Keras' built-in `class_weight` doesn't support multi-label directly —
  a custom weighted binary_crossentropy would be the fix) is the next thing worth trying.
- **Exact match ratio vs Hamming loss:** exact match is stricter (all labels must be right),
  hamming loss is more forgiving (rewards partial credit per label). Report both, since only
  exact match tends to make results look worse than they are for a 3-label problem.
- If results still look weak specifically on blast-containing composites, that's a signal the model
  needs either more blast training examples, or that blast lesions get visually "overwhelmed" in
  composite images by the other disease's lesions — Grad-CAM on a few composite images would confirm
  which regions the model is actually attending to.